# FedKG-Maint — Phase 2: Centralized baselines and the evaluation toolkit

**Input:** the verified data of Phase 1 (`prep_v2/`). **No federated learning, no privacy, no KG/LLM here.**

## What this phase answers
1. What do trivial baselines achieve (prevalence, "time since last error", "time since last maintenance")? Any model must beat them.
2. How well do centralized Logistic Regression and Gradient Boosting do — the **upper-reference** that federated models will be compared with?
3. How much does `maintenance_recency` contribute (`full` vs `no_maintenance`)?
4. Does the normalization policy (`global` vs `local`) change the result?
5. How uncertain are all these numbers? (cluster bootstrap over **machines**, not rows).

## Evaluation protocol (fixed before any model is trained)
| Eval set | Used for | Notes |
|---|---|---|
| `validation` (factories 1–4) | model selection and the alert threshold — **the only set used for selection** | few events (46): selection is noisy |
| `test_site_post` (factory_5, after the training window) | **primary held-out metric**; the only set where `global` and `local` can be compared fairly | no overlap with any scaler calibration window |
| `test_time_late` (factories 1–4, late window) | same sites, future time | isolates the time effect |
| `test_site_late` (factory_5, late window) | strictest test, only 13 events | very wide intervals |
| `test_site_full` (all of factory_5) | most events; **`global` only** — under `local`, half of its rows calibrated the scaler | flagged in every table |

Primary metric: **PR-AUC (average precision)** with its lift over prevalence, plus ROC-AUC and, at a validation-chosen threshold, precision / recall / F1 / false-positive rate / **event recall** (share of failure events preceded by at least one alert in the 24 h before). Confidence intervals: cluster bootstrap over machines; model comparisons use the **same** resamples (paired).

In [ ]:
# 1.1  Setup, single CONFIG, directories, check registry
import os, sys, json, time, importlib, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import joblib
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings("ignore", category=ConvergenceWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

try:  # Google Colab
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = Path("/content/drive/MyDrive/FedKG-Maint")
except ImportError:  # local run / testing
    PROJECT_DIR = Path(os.environ.get("FEDKG_ROOT", "./FedKG-Maint")).resolve()

CONFIG = {
    "SEED": 42,
    "BOOTSTRAP_B": 1000,                      # machine-level bootstrap replicates
    "CI_LEVEL": 0.95,
    "EVAL_SETS": ["validation", "test_time_late", "test_site_post", "test_site_late", "test_site_full"],
    "POLICIES": ["global", "local"],
    "FEATURE_SETS": ["full", "no_maintenance"],
    "LR_GRID_C": [0.01, 0.1, 1.0],
    "HGB_GRID": [dict(learning_rate=0.1, max_depth=3, max_iter=150), dict(learning_rate=0.1, max_depth=6, max_iter=150),
                 dict(learning_rate=0.05, max_depth=3, max_iter=300), dict(learning_rate=0.05, max_depth=6, max_iter=300)],
    "FRAGILE_EVENTS": 30,
}
if os.environ.get("FEDKG_FAST"):              # quick smoke test only (used to test this notebook)
    CONFIG.update(BOOTSTRAP_B=60, LR_GRID_C=[0.1], HGB_GRID=[dict(learning_rate=0.1, max_depth=3, max_iter=40)])

PREP_DIR = PROJECT_DIR / "prep_v2"
RAW_DIR = PROJECT_DIR / "data" / "raw"
P2_DIR = PROJECT_DIR / "phase2_baselines"
for d in ["results", "predictions", "models", "reports"]:
    (P2_DIR / d).mkdir(parents=True, exist_ok=True)
assert (PREP_DIR / "processed" / "train.parquet").exists(), "Run the Phase-1 notebook first (prep_v2 not found)"

sys.path.insert(0, str(PREP_DIR))
import fedkg_data
FEATURE_CFG = json.loads((PREP_DIR / "configs" / "feature_groups.json").read_text())
ALL_FEATURES = FEATURE_CFG["feature_sets"]["full"]
TRAIN_CUTOFF = pd.Timestamp(FEATURE_CFG["boundaries"]["train_cutoff"])
np.random.seed(CONFIG["SEED"])

failures = pd.read_csv(RAW_DIR / "PdM_failures.csv", parse_dates=["datetime"])
failures["datetime"] = failures["datetime"].astype("datetime64[ns]")
fail_events = (failures[["machineID", "datetime"]].drop_duplicates()
               .sort_values(["machineID", "datetime"]).reset_index(drop=True))

CHECKS = {}
def record(check_id, ok, detail=""):
    CHECKS[check_id] = {"status": "PASS" if ok else "FAIL", "detail": str(detail)}
    print(f"[{'PASS' if ok else 'FAIL'}] {check_id}: {detail}")
    if not ok:
        raise AssertionError(f"Check failed: {check_id} — {detail}")

print("pandas", pd.__version__, "| numpy", np.__version__, "| project:", PROJECT_DIR)
print("Failure events (distinct machine, timestamp):", len(fail_events))

## 2. Evaluation toolkit (written to `phase2_baselines/fedkg_eval.py`, reused by every later phase)

Metrics are computed from **machine weights**: a bootstrap replicate is just an integer weight per machine (how many times it was drawn), so thousands of replicates are cheap and every comparison between two models can use the identical resamples. The implementation handles tied scores exactly like scikit-learn and is verified against it (and against physically replicated rows) before any model is trained.

In [ ]:
# 2.1  Write the evaluation module, import it, and verify it against scikit-learn
MODULE_SRC = '''"""Evaluation toolkit for FedKG-Maint (machine-level weighted metrics + cluster bootstrap).

    import sys; sys.path.insert(0, "<PROJECT_DIR>/phase2_baselines")
    from fedkg_eval import SetEval, draw_weights
    se = SetEval(meta, y, fail_events)          # meta: DataFrame with machineID, datetime sorted by (machine, time)
    pk = se.pack(scores); thr = se.best_f1_threshold(pk)       # threshold: fit on VALIDATION only
    point = se.metrics(pk, None, thr)
    reps = se.bootstrap(pk, thr, draw_weights(se.M, 1000, seed))
"""
import numpy as np
import pandas as pd

METRIC_COLS = ["ap", "roc_auc", "precision", "recall", "f1", "fpr", "event_recall"]


def draw_weights(n_machines, B, seed):
    """B bootstrap resamples of machines: integer counts per machine (rows sum to n_machines)."""
    rng = np.random.default_rng(seed)
    return rng.multinomial(n_machines, np.full(n_machines, 1.0 / n_machines), size=B).astype("float64")


def ci(values, level=0.95):
    a = (1.0 - level) / 2.0
    return tuple(np.percentile(np.asarray(values, dtype="float64"), [100 * a, 100 * (1 - a)]))


class SetEval:
    def __init__(self, meta, y, fail_events, horizon_hours=24):
        self.y = np.asarray(y, dtype="float64")
        self.n = len(self.y)
        mid = meta["machineID"].to_numpy()
        self.machines, self.mach_idx = np.unique(mid, return_inverse=True)
        self.mach_idx = self.mach_idx.ravel()
        self.M = len(self.machines)
        ts = meta["datetime"].to_numpy()
        assert np.all(np.diff(self.mach_idx) >= 0), "rows must be sorted by machine"
        assert np.all((np.diff(ts.astype("int64")) >= 0) | (np.diff(self.mach_idx) > 0)), "rows must be sorted by time inside a machine"
        starts = np.searchsorted(self.mach_idx, np.arange(self.M), side="left")
        ends = np.searchsorted(self.mach_idx, np.arange(self.M), side="right")
        H = np.timedelta64(int(horizon_hours), "h")
        slices, ev_m = [], []
        fe = fail_events[fail_events["machineID"].isin(self.machines)]
        for m, g in fe.groupby("machineID"):
            k = int(np.searchsorted(self.machines, m))
            s, e = int(starts[k]), int(ends[k])
            t_m = ts[s:e]
            f = g["datetime"].to_numpy()
            lo = np.searchsorted(t_m, f - H, side="left")      # rows with t >= f - 24h
            hi = np.searchsorted(t_m, f, side="left")          # rows with t <  f
            for a, b in zip(lo, hi):
                if b > a:
                    slices.append((s + int(a), s + int(b)))
                    ev_m.append(k)
        self.ev_slices = slices
        self.ev_mach = np.array(ev_m, dtype=int)
        self.n_events = len(slices)

    def pack(self, score):
        score = np.asarray(score, dtype="float64")
        if not np.isfinite(score).all():
            raise ValueError("non-finite scores")
        uniq, inv = np.unique(-score, return_inverse=True)     # group 0 = highest score; ties share a group
        return {"score": score, "inv": np.asarray(inv).ravel().astype(np.int64), "G": len(uniq), "thr_vals": -uniq}

    def best_f1_threshold(self, pk):
        tp_g = np.bincount(pk["inv"], self.y, pk["G"])
        fp_g = np.bincount(pk["inv"], 1.0 - self.y, pk["G"])
        tp, fp = np.cumsum(tp_g), np.cumsum(fp_g)
        f1 = 2.0 * tp / (tp + fp + tp[-1])
        return float(pk["thr_vals"][int(np.argmax(f1))])

    def _event_flags(self, pk, thr):
        if self.n_events == 0:
            return np.zeros(0, dtype=bool)
        if "ev_max" not in pk:
            pk["ev_max"] = np.array([pk["score"][a:b].max() for a, b in self.ev_slices])
        return pk["ev_max"] >= thr

    def metrics(self, pk, wm=None, thr=None):
        wr = np.ones(self.n) if wm is None else wm[self.mach_idx]
        wy = wr * self.y
        wn = wr - wy
        tp_g = np.bincount(pk["inv"], wy, pk["G"])
        fp_g = np.bincount(pk["inv"], wn, pk["G"])
        tp, fp = np.cumsum(tp_g), np.cumsum(fp_g)
        P, N = float(tp[-1]), float(fp[-1])
        out = {"prevalence": P / (P + N) if P + N > 0 else np.nan}
        if P <= 0 or N <= 0:
            out.update({k: np.nan for k in METRIC_COLS})
            return out
        denom = tp + fp
        prec = np.divide(tp, denom, out=np.zeros_like(tp), where=denom > 0)
        rec = tp / P
        out["ap"] = float(np.sum(np.diff(np.concatenate(([0.0], rec))) * prec))
        out["roc_auc"] = float(np.sum(tp_g * (N - fp + 0.5 * fp_g)) / (P * N))
        if thr is not None:
            pred = pk["score"] >= thr
            tpt, fpt = float(np.sum(wy * pred)), float(np.sum(wn * pred))
            p_ = tpt / (tpt + fpt) if tpt + fpt > 0 else 0.0
            r_ = tpt / P
            out.update({"precision": p_, "recall": r_, "f1": 2 * p_ * r_ / (p_ + r_) if p_ + r_ > 0 else 0.0, "fpr": fpt / N})
            wev = np.ones(self.n_events) if wm is None else wm[self.ev_mach]
            flag = self._event_flags(pk, thr)
            out["event_recall"] = float(np.sum(wev * flag) / np.sum(wev)) if np.sum(wev) > 0 else np.nan
        return out

    def bootstrap(self, pk, thr, W):
        return pd.DataFrame([self.metrics(pk, w, thr) for w in W])
'''
(P2_DIR / "fedkg_eval.py").write_text(MODULE_SRC)
sys.path.insert(0, str(P2_DIR))
import fedkg_eval
importlib.reload(fedkg_eval)
from fedkg_eval import SetEval, draw_weights, ci

# ---- verification against scikit-learn (with tied scores) and against physically replicated rows
rng = np.random.default_rng(CONFIG["SEED"])
n_m, per = 20, 200
meta_t = pd.DataFrame({"machineID": np.repeat(np.arange(1, n_m + 1), per),
                       "datetime": np.tile(pd.date_range("2015-01-01", periods=per, freq="h").to_numpy(), n_m)})
y_t = (rng.random(n_m * per) < 0.08).astype(float)
s_t = np.round(1.5 * y_t + rng.normal(0, 1, n_m * per), 1)                       # many ties on purpose
no_events = pd.DataFrame({"machineID": pd.Series(dtype="int64"), "datetime": pd.Series(dtype="datetime64[ns]")})
se_t = SetEval(meta_t, y_t, no_events)
m0 = se_t.metrics(se_t.pack(s_t))
ok_sk = abs(m0["ap"] - average_precision_score(y_t, s_t)) < 1e-9 and abs(m0["roc_auc"] - roc_auc_score(y_t, s_t)) < 1e-9
counts = rng.multinomial(n_m, np.full(n_m, 1 / n_m))
rows_rep = np.repeat(np.arange(len(y_t)), counts[se_t.mach_idx].astype(int))
mw = se_t.metrics(se_t.pack(s_t), counts.astype(float))
ok_rep = (abs(mw["ap"] - average_precision_score(y_t[rows_rep], s_t[rows_rep])) < 1e-9
          and abs(mw["roc_auc"] - roc_auc_score(y_t[rows_rep], s_t[rows_rep])) < 1e-9)
record("E-1 weighted AP / ROC-AUC equal scikit-learn (ties) and equal metrics on physically replicated rows", bool(ok_sk and ok_rep),
       f"AP={m0['ap']:.6f} vs sklearn {average_precision_score(y_t, s_t):.6f}")

# event window: failure at hour 50 -> rows t with t < f <= t + 24h  =>  rows 26..49
meta_e = meta_t[meta_t["machineID"] == 1].reset_index(drop=True)
fe_t = pd.DataFrame({"machineID": [1], "datetime": [meta_e["datetime"].iloc[50]]})
se_e = SetEval(meta_e, np.zeros(len(meta_e)), fe_t)
record("E-2 event window = the 24 rows strictly before the failure", se_e.ev_slices == [(26, 50)], f"slice={se_e.ev_slices}")

zero_w = se_t.metrics(se_t.pack(s_t), np.r_[np.zeros(n_m - 1), n_m].astype(float))   # one machine only: must not crash
print("Toolkit written to", P2_DIR / "fedkg_eval.py")

## 3. Load the data and define the evaluation sets

Every (policy, feature-set) combination is loaded through the Phase-1 loader. `test_site_post` is derived here (factory_5 rows after the training window) — no file in `prep_v2/` is created or changed.

In [ ]:
# 3.1  Load all data combinations and build the evaluation sets
BASE_SPLITS = ["validation", "test_time_late", "test_site_late", "test_site_full"]
DATA = {}                                          # (policy, feature_set) -> {"train": (X, y), eval_set: X}
META_BASE, Y_BASE = {}, {}
for policy in CONFIG["POLICIES"]:
    for fs in CONFIG["FEATURE_SETS"]:
        d = {}
        Xtr, ytr, mtr = fedkg_data.load_split("train", policy=policy, feature_set=fs)
        d["train"] = (Xtr, ytr)
        for s in BASE_SPLITS:
            X, y, meta = fedkg_data.load_split(s, policy=policy, feature_set=fs)
            d[s] = X
            META_BASE.setdefault(s, meta); Y_BASE.setdefault(s, y)
        post_mask = (META_BASE["test_site_full"]["datetime"] >= TRAIN_CUTOFF).to_numpy()
        d["test_site_post"] = d["test_site_full"][post_mask]
        DATA[(policy, fs)] = d

post_mask = (META_BASE["test_site_full"]["datetime"] >= TRAIN_CUTOFF).to_numpy()
EVAL = {}
for s in CONFIG["EVAL_SETS"]:
    meta = META_BASE["test_site_full"][post_mask].reset_index(drop=True) if s == "test_site_post" else META_BASE[s]
    y = Y_BASE["test_site_full"][post_mask] if s == "test_site_post" else Y_BASE[s]
    EVAL[s] = {"meta": meta, "y": y, "se": SetEval(meta, y, fail_events)}

rows = []
for s, e in EVAL.items():
    rows.append({"eval_set": s, "rows": len(e["y"]), "machines": e["se"].M, "positive_rows": int(e["y"].sum()),
                 "prevalence_%": round(100 * e["y"].mean(), 3), "failure_events": e["se"].n_events,
                 "fragile": e["se"].n_events < CONFIG["FRAGILE_EVENTS"]})
EVAL_TABLE = pd.DataFrame(rows)
print(EVAL_TABLE.to_string(index=False))

record("D-1 held-out sets contain only factory_5 machines; validation/time-test contain none of them",
       all(set(EVAL[s]["meta"]["client_id"]) == {"factory_5"} for s in ["test_site_post", "test_site_late", "test_site_full"])
       and all("factory_5" not in set(EVAL[s]["meta"]["client_id"]) for s in ["validation", "test_time_late"]))
post_min = EVAL["test_site_post"]["meta"]["datetime"].min()
record("D-2 test_site_post starts at/after the end of the scaler calibration window (no overlap with local_factory_5 fitting)",
       post_min >= TRAIN_CUTOFF, f"first row {post_min} >= {TRAIN_CUTOFF}")

## 4. Reference baselines (no learning)

* **prevalence** — a constant score: PR-AUC equals the positive rate, ROC-AUC 0.5. The floor every model must beat.
* **error recency** — risk is higher when the last error was more recent.
* **maintenance recency** — risk is higher when the last maintenance record was more recent (partly "time since the last failure", see Phase 1, D-2).

All three are scored with the same toolkit and the same validation-chosen threshold rule as the models.

In [ ]:
# 4.1  Registry of scored models + the three reference baselines
SCORES, INFO, SCORING_LOG, SEL_ROWS = {}, {}, [], []     # SCORING_LOG records every (candidate, split) ever scored during selection

def register(key, model, fs, policy, params, score_by_set):
    SCORES[key] = score_by_set
    INFO[key] = {"model": model, "feature_set": fs, "policy": policy, "params": params}

# unscaled features for the heuristics
RAW = {}
for s in BASE_SPLITS:
    RAW[s] = fedkg_data.load_split(s, policy="none", feature_set="full")[0]
RAW["test_site_post"] = RAW["test_site_full"][post_mask]
col = {c: i for i, c in enumerate(ALL_FEATURES)}

register("prevalence|full|none", "prevalence", "full", "none", {}, {s: np.zeros(len(EVAL[s]["y"])) for s in CONFIG["EVAL_SETS"]})
register("heur_error_recency|full|none", "heur_error_recency", "full", "none", {"feature": "hours_since_last_error_log1p"},
         {s: -RAW[s][:, col["hours_since_last_error_log1p"]].astype("float64") for s in CONFIG["EVAL_SETS"]})
register("heur_maint_recency|full|none", "heur_maint_recency", "full", "none", {"feature": "hours_since_last_maintenance_log1p"},
         {s: -RAW[s][:, col["hours_since_last_maintenance_log1p"]].astype("float64") for s in CONFIG["EVAL_SETS"]})
print("Reference baselines registered:", [k for k in SCORES])

## 5. Centralized models: Logistic Regression and Gradient Boosting

Trained on the pooled training rows of factories 1–4 (the centralized reference). For each of the 4 combinations of feature set (`full`, `no_maintenance`) and normalization policy (`global`, `local`), the hyper-parameters are chosen **only by validation PR-AUC**; test sets are scored afterwards, once, for the selected model. Class imbalance is handled with balanced class weights. The validation set has few failure events, so the selection is noisy — the grids are small on purpose.

In [ ]:
# 5.1  Train, select on validation only, then score every evaluation set once
MODELS = {}
val_se = EVAL["validation"]["se"]
def val_ap(score):
    return val_se.metrics(val_se.pack(score))["ap"]

def select_and_register(family, fit_fn, score_fn, grid, policy, fs):
    Xtr, ytr = DATA[(policy, fs)]["train"]
    Xva = DATA[(policy, fs)]["validation"]
    best = None
    for params in grid:
        t0 = time.time()
        model = fit_fn(params).fit(Xtr, ytr)
        SCORING_LOG.append((f"{family}|{fs}|{policy}|{params}", "validation"))      # only validation is scored during selection
        ap = val_ap(score_fn(model, Xva))
        SEL_ROWS.append({"family": family, "feature_set": fs, "policy": policy, "params": str(params),
                         "val_ap": round(ap, 5), "seconds": round(time.time() - t0, 1)})
        if best is None or ap > best[0]:
            best = (ap, params, model)
    ap, params, model = best
    key = f"{family}|{fs}|{policy}"
    MODELS[key] = model
    # selection is finished: only now are the remaining evaluation sets scored
    register(key, family, fs, policy, params, {s: score_fn(model, DATA[(policy, fs)][s]) for s in CONFIG["EVAL_SETS"]})
    print(f"  {key:28s} best val AP={ap:.4f}  params={params}")

lr_fit = lambda p: LogisticRegression(C=p["C"], class_weight="balanced", max_iter=500)
lr_score = lambda m, X: m.decision_function(X)
hgb_fit = lambda p: HistGradientBoostingClassifier(learning_rate=p["learning_rate"], max_depth=p["max_depth"], max_iter=p["max_iter"],
                                                   l2_regularization=1.0, class_weight="balanced", early_stopping=False,
                                                   random_state=CONFIG["SEED"])
hgb_score = lambda m, X: m.predict_proba(X)[:, 1]

for policy in CONFIG["POLICIES"]:
    for fs in CONFIG["FEATURE_SETS"]:
        print(f"[{policy} / {fs}]")
        select_and_register("logreg", lr_fit, lr_score, [{"C": c} for c in CONFIG["LR_GRID_C"]], policy, fs)
        select_and_register("hgb", hgb_fit, hgb_score, CONFIG["HGB_GRID"], policy, fs)

SELECTION = pd.DataFrame(SEL_ROWS)
SELECTION.to_csv(P2_DIR / "results" / "model_selection_validation.csv", index=False)
print(f"\n{len(SELECTION)} candidate fits; selection log saved.")

## 6. Point estimates and machine-level bootstrap

Thresholds (F1-optimal) are fitted on the validation scores only and then frozen. For each evaluation set, **one** set of bootstrap weights is drawn and reused for every model, so differences between models are paired. The prevalence-based `lift` is `PR-AUC / positive rate`.

In [ ]:
# 6.1  Thresholds from validation, point estimates and paired cluster bootstrap for every model x eval set
B, LEVEL = CONFIG["BOOTSTRAP_B"], CONFIG["CI_LEVEL"]
W = {s: draw_weights(EVAL[s]["se"].M, B, CONFIG["SEED"] + 17 * i) for i, s in enumerate(CONFIG["EVAL_SETS"])}   # shared across models

THR, POINT, REPS = {}, {}, {}
t0 = time.time()
for key in SCORES:
    pk_val = val_se.pack(SCORES[key]["validation"])
    THR[key] = val_se.best_f1_threshold(pk_val)                                   # validation only
    for s in CONFIG["EVAL_SETS"]:
        se = EVAL[s]["se"]
        pk = se.pack(SCORES[key][s])
        POINT[(key, s)] = se.metrics(pk, None, THR[key])
        reps = se.bootstrap(pk, THR[key], W[s])
        POINT[(key, s)]["lift"] = POINT[(key, s)]["ap"] / POINT[(key, s)]["prevalence"]
        reps["lift"] = reps["ap"] / reps["prevalence"]
        REPS[(key, s)] = reps
    print(f"  {key:30s} done ({time.time() - t0:.0f}s)")

rows = []
for (key, s), p in POINT.items():
    info = INFO[key]
    for mname in ["ap", "lift", "roc_auc", "precision", "recall", "f1", "fpr", "event_recall"]:
        lo, hi = ci(REPS[(key, s)][mname].dropna(), LEVEL)
        rows.append({"model_key": key, "model": info["model"], "feature_set": info["feature_set"], "policy": info["policy"],
                     "eval_set": s, "metric": mname, "value": p[mname], "ci_low": lo, "ci_high": hi,
                     "events": EVAL[s]["se"].n_events, "machines": EVAL[s]["se"].M,
                     "calibration_overlap": bool(info["policy"] == "local" and s == "test_site_full")})
RESULTS = pd.DataFrame(rows)
RESULTS.to_csv(P2_DIR / "results" / "baseline_results_long.csv", index=False)
print(f"\nResults table: {len(RESULTS):,} rows saved. Bootstrap: B={B}, level={LEVEL:.0%}, resampling machines.")

## 7. Results

Read the tables with three caveats in mind: (1) intervals resample **machines**, the correct unit of independence; (2) sets with fewer than 30 failure events are flagged — their intervals are wide by nature; (3) `test_site_full` under the `local` policy is excluded from comparisons because half of its rows calibrated that scaler.

In [ ]:
# 7.1  Ranked tables (primary metric: PR-AUC) for the key evaluation sets
def fmt(v, lo, hi): return f"{v:.3f} [{lo:.3f}, {hi:.3f}]"
def table(eval_set):
    d = RESULTS[(RESULTS["eval_set"] == eval_set) & (~RESULTS["calibration_overlap"])]
    piv = d.pivot_table(index="model_key", columns="metric", values=["value", "ci_low", "ci_high"])
    out = pd.DataFrame({
        "PR-AUC [95% CI]": [fmt(piv.loc[k, ("value", "ap")], piv.loc[k, ("ci_low", "ap")], piv.loc[k, ("ci_high", "ap")]) for k in piv.index],
        "lift": piv[("value", "lift")].round(2).to_numpy(),
        "ROC-AUC": piv[("value", "roc_auc")].round(3).to_numpy(),
        "event recall": piv[("value", "event_recall")].round(2).to_numpy(),
        "FPR": piv[("value", "fpr")].round(3).to_numpy(),
        "_ap": piv[("value", "ap")].to_numpy()}, index=piv.index)
    return out.sort_values("_ap", ascending=False).drop(columns="_ap")

for s in ["test_site_post", "test_time_late", "validation"]:
    e = EVAL[s]["se"]
    note = "  <-- FRAGILE (few events)" if e.n_events < CONFIG["FRAGILE_EVENTS"] else ""
    print(f"\n=== {s}: {e.M} machines, {e.n_events} failure events, prevalence {100 * EVAL[s]['y'].mean():.2f}%{note} ===")
    print(table(s).to_string())
print("\n(test_site_late and test_site_full are in results/baseline_results_long.csv; test_site_late has only"
      f" {EVAL['test_site_late']['se'].n_events} events.)")

In [ ]:
# 7.2  Paired comparisons (same bootstrap resamples): maintenance ablation, normalization policy, model vs heuristics
def paired(a, b, s, metric="ap"):
    d = REPS[(a, s)][metric].to_numpy() - REPS[(b, s)][metric].to_numpy()
    lo, hi = ci(d, LEVEL)
    return POINT[(a, s)][metric] - POINT[(b, s)][metric], lo, hi

MATERIAL = 0.005          # a PR-AUC difference below half a percentage point is treated as negligible in practice
def verdict(lo, hi, d):
    if not (lo > 0 or hi < 0):
        return "no clear difference"
    return "differs (CI excludes 0)" if abs(d) >= MATERIAL else "CI excludes 0 but negligible (<0.005)"

CMP_ROWS = []
def add(kind, a, b, s):
    d, lo, hi = paired(a, b, s)
    CMP_ROWS.append({"comparison": kind, "A": a, "B": b, "eval_set": s, "delta_AP(A-B)": round(d, 4),
                     "ci_low": round(lo, 4), "ci_high": round(hi, 4), "verdict": verdict(lo, hi, d)})

fair_sets = ["test_site_post", "test_time_late", "test_site_late", "validation"]
for fam in ["logreg", "hgb"]:
    for pol in CONFIG["POLICIES"]:
        for s in fair_sets:
            add("maintenance ablation (full - no_maintenance)", f"{fam}|full|{pol}", f"{fam}|no_maintenance|{pol}", s)
    for fs in CONFIG["FEATURE_SETS"]:
        for s in fair_sets:
            add("normalization (global - local)", f"{fam}|{fs}|global", f"{fam}|{fs}|local", s)
best_heur = max(["heur_error_recency|full|none", "heur_maint_recency|full|none"], key=lambda k: POINT[(k, "test_site_post")]["ap"])
for fam in ["logreg", "hgb"]:
    for s in ["test_site_post", "test_time_late"]:
        add(f"model vs best heuristic ({best_heur.split('|')[0]})", f"{fam}|full|global", best_heur, s)
CMP = pd.DataFrame(CMP_ROWS)
CMP.to_csv(P2_DIR / "results" / "paired_comparisons.csv", index=False)
with pd.option_context("display.width", 250, "display.max_colwidth", 60):
    for kind in CMP["comparison"].unique():
        print(f"\n--- {kind} ---")
        print(CMP[CMP["comparison"] == kind].drop(columns="comparison").to_string(index=False))

print(f"\nNote: {len(CMP)} paired comparisons were made. About 1 in 20 would exclude 0 by chance alone, so rely on "
      "patterns that repeat across evaluation sets and models (test_site_post and test_time_late first), not on a single flagged row.")

In [ ]:
# 8.1  Integrity checks, saved artifacts, final report
# M-1: during model selection only the validation set was ever scored
record("M-1 model selection used the validation set only", all(sp == "validation" for _, sp in SCORING_LOG),
       f"{len(SCORING_LOG)} candidate fits, all scored on validation only")

# M-2: thresholds were fitted on validation and are reproducible
ok_thr = all(abs(THR[k] - val_se.best_f1_threshold(val_se.pack(SCORES[k]["validation"]))) < 1e-12 for k in SCORES)
record("M-2 alert thresholds derive from validation scores only", ok_thr, f"{len(THR)} thresholds")

# M-3: the constant (prevalence) baseline has AP == prevalence and ROC-AUC == 0.5 on every set
ok_prev = all(abs(POINT[("prevalence|full|none", s)]["ap"] - POINT[("prevalence|full|none", s)]["prevalence"]) < 1e-12
              and abs(POINT[("prevalence|full|none", s)]["roc_auc"] - 0.5) < 1e-12 for s in CONFIG["EVAL_SETS"])
record("M-3 prevalence baseline: PR-AUC = positive rate, ROC-AUC = 0.5", ok_prev)

# M-4: point estimates lie inside their bootstrap intervals (sanity of the bootstrap)
inside = RESULTS[RESULTS["metric"] == "ap"].apply(lambda r: r["ci_low"] - 1e-9 <= r["value"] <= r["ci_high"] + 1e-9, axis=1)
record("M-4 AP point estimates fall inside their bootstrap intervals", bool(inside.mean() > 0.9), f"{inside.mean():.0%} of {len(inside)} intervals")

# save predictions (base splits only; post/late are derivable by datetime) and the selected models
for key in SCORES:
    parts = []
    for s in ["validation", "test_time_late", "test_site_full"]:
        m = EVAL[s]["meta"][["machineID", "datetime"]].copy()
        m["eval_set"], m["score"] = s, SCORES[key][s].astype("float32")
        parts.append(m)
    pd.concat(parts, ignore_index=True).to_parquet(P2_DIR / "predictions" / (key.replace("|", "__") + ".parquet"), index=False)
for key, mdl in MODELS.items():
    joblib.dump(mdl, P2_DIR / "models" / (key.replace("|", "__") + ".joblib"))

best_key = max((k for k in SCORES if INFO[k]["model"] in ("logreg", "hgb")), key=lambda k: POINT[(k, "validation")]["ap"])
REFERENCES = {}                      # centralized reference per (feature_set, policy): better of logreg/hgb on VALIDATION
for fs in CONFIG["FEATURE_SETS"]:
    for pol in CONFIG["POLICIES"]:
        REFERENCES[f"{fs}|{pol}"] = max([f"logreg|{fs}|{pol}", f"hgb|{fs}|{pol}"], key=lambda k: POINT[(k, "validation")]["ap"])
rep = {
    "config": {k: v for k, v in CONFIG.items()},
    "eval_sets": EVAL_TABLE.to_dict("records"),
    "selected_hyperparameters": {k: INFO[k]["params"] for k in SCORES},
    "thresholds_from_validation": THR,
    "best_model_by_validation_ap": best_key,
    "centralized_reference_per_feature_set_and_policy": REFERENCES,
    "headline_primary_set": {k: {"ap": POINT[(k, "test_site_post")]["ap"], "lift": POINT[(k, "test_site_post")]["lift"],
                                 "ci": ci(REPS[(k, "test_site_post")]["ap"], LEVEL)} for k in SCORES},
    "checks": CHECKS,
    "limitations": [
        "Selection and thresholds rely on a validation set with few failure events; small differences between candidates are not meaningful.",
        "Bootstrap intervals resample machines (20 in the held-out factory): they reflect machine-level variability, not variability of the factory partition.",
        "test_site_late has very few events; test_site_full is excluded for the local policy because of scaler-calibration overlap.",
        "Single training seed for the tree/linear models; the dominant uncertainty is the evaluation sample, which the bootstrap covers.",
        "Centralized models pool training rows of the four factories: this is the reference, not a federated result."],
}
(P2_DIR / "reports" / "phase2_baselines_report.json").write_text(json.dumps(rep, indent=2, default=str))

pr = lambda k, s: f"{POINT[(k, s)]['ap']:.3f} [{ci(REPS[(k, s)]['ap'], LEVEL)[0]:.3f}, {ci(REPS[(k, s)]['ap'], LEVEL)[1]:.3f}]"
prev = POINT[("prevalence|full|none", "test_site_post")]["prevalence"]
print("\n================ READOUT (primary held-out set: test_site_post) ================")
print(f"Prevalence (random baseline PR-AUC): {prev:.3f}")
for k in ["heur_error_recency|full|none", "heur_maint_recency|full|none", f"logreg|full|global", f"hgb|full|global",
          f"logreg|no_maintenance|global", f"hgb|no_maintenance|global"]:
    print(f"  {k:30s} PR-AUC {pr(k, 'test_site_post')}")
print("\nNumber to carry into Phase 3 (federated models are compared with the centralized reference):")
print("  centralized reference per (feature set | policy) = better of logreg/hgb on VALIDATION PR-AUC:")
for combo, k in REFERENCES.items():
    print(f"    {combo:22s} -> {k:30s} test_site_post PR-AUC {pr(k, 'test_site_post')}")
print("  A federated model must be compared with the reference of its OWN feature set and policy.")
print("\nPHASE 2 BASELINES COMPLETE — all checks PASS.")

In [ ]:
# D1. Corrected orientation of the two heuristic baselines (sign chosen on VALIDATION only)
from sklearn.metrics import roc_auc_score
col = {c: i for i, c in enumerate(ALL_FEATURES)}
RAW_TRAIN = fedkg_data.load_split("train", policy="none", feature_set="full")[0]
rows = []
for name, feat in [("error recency", "hours_since_last_error_log1p"), ("maintenance recency", "hours_since_last_maintenance_log1p")]:
    auc_neg = roc_auc_score(Y_BASE["validation"], -RAW["validation"][:, col[feat]])     # assumption used in Phase 2: recent => risky
    sign = -1.0 if auc_neg >= 0.5 else 1.0
    row = {"heuristic": name, "val_ROC_AUC_if_recent_is_risky": round(auc_neg, 3),
           "chosen_direction": "recent => risky" if sign < 0 else "LONG time since => risky"}
    for s in ["validation", "test_time_late", "test_site_post"]:
        se = EVAL[s]["se"]
        row[f"AP_{s}"] = round(se.metrics(se.pack(sign * RAW[s][:, col[feat]].astype("float64")))["ap"], 3)
    rows.append(row)
print(pd.DataFrame(rows).to_string(index=False))
print("\nIf 'maintenance recency' is oriented 'LONG time since => risky', the Phase-2 reference row for it was inverted (AP below prevalence).")

In [ ]:
# D2. Gradient boosting on single feature groups and combinations (same hyper-parameters as the selected hgb|full|global)
GROUPS = FEATURE_CFG["groups"]
Xtr_g, ytr_g = DATA[("global", "full")]["train"]
best_p = INFO["hgb|full|global"]["params"]
def fit_ap(idx, Xtr, Xev):
    m = HistGradientBoostingClassifier(learning_rate=best_p["learning_rate"], max_depth=best_p["max_depth"], max_iter=best_p["max_iter"],
                                       l2_regularization=1.0, class_weight="balanced", early_stopping=False,
                                       random_state=CONFIG["SEED"]).fit(Xtr[:, idx], ytr_g)
    out = {}
    for s in ["validation", "test_time_late", "test_site_post"]:
        se = EVAL[s]["se"]
        out[s] = se.metrics(se.pack(m.predict_proba(Xev[s][:, idx])[:, 1]))["ap"]
    return out

def idx_of(*groups): return [col[c] for g in groups for c in GROUPS[g]]
combos = {
    "sensor_raw only": idx_of("sensor_raw"),
    "rolling_stats only": idx_of("rolling_stats"),
    "sensors + rolling": idx_of("sensor_raw", "rolling_stats"),
    "error_recency only": idx_of("error_recency"),
    "maintenance_recency only": idx_of("maintenance_recency"),
    "history_counts only": idx_of("history_counts"),
    "maintenance + errors": idx_of("maintenance_recency", "error_recency"),
    "maintenance + sensors + rolling": idx_of("maintenance_recency", "sensor_raw", "rolling_stats"),
    "all (reference)": list(range(len(ALL_FEATURES))),
}
Xev_g = DATA[("global", "full")]
rows = []
for name, idx in combos.items():
    r = fit_ap(idx, Xtr_g, Xev_g)
    rows.append({"features": name, "n": len(idx), **{f"AP_{k}": round(v, 3) for k, v in r.items()}})
    print(f"  {name:34s} AP validation={r['validation']:.3f}  time_late={r['test_time_late']:.3f}  site_post={r['test_site_post']:.3f}")
GROUP_ABLATION = pd.DataFrame(rows)
GROUP_ABLATION.to_csv(P2_DIR / "results" / "diagnostic_group_ablation.csv", index=False)

In [ ]:
# D3. Timing structure of failures relative to maintenance records
maint_raw = pd.read_csv(RAW_DIR / "PdM_maint.csv", parse_dates=["datetime"]).assign(datetime=lambda d: d["datetime"].astype("datetime64[ns]"))
errors_raw = pd.read_csv(RAW_DIR / "PdM_errors.csv", parse_dates=["datetime"]).assign(datetime=lambda d: d["datetime"].astype("datetime64[ns]"))

print("Hour of day of the records (count by hour; a single dominant hour = daily batch logging):")
for name, df in [("failures", failures), ("maintenance", maint_raw), ("errors", errors_raw)]:
    h = df["datetime"].dt.hour.value_counts().sort_index()
    print(f"  {name:12s} top hours: {h.sort_values(ascending=False).head(4).to_dict()}   (n={len(df)})")

# age of the machine's last maintenance at each failure, and age of the SAME component's last replacement
f = failures.rename(columns={"failure": "comp"}).sort_values("datetime").reset_index(drop=True)
mm = maint_raw.rename(columns={"datetime": "m_time"}).assign(datetime=lambda d: d["m_time"]).sort_values("datetime")
any_m = pd.merge_asof(f, mm[["machineID", "datetime", "m_time"]].drop_duplicates(["machineID", "datetime"]), on="datetime", by="machineID",
                      direction="backward", allow_exact_matches=False)
f["age_any_h"] = (f["datetime"] - any_m["m_time"]).dt.total_seconds().to_numpy() / 3600
comp_m = pd.merge_asof(f[["machineID", "comp", "datetime"]], mm, on="datetime", by=["machineID", "comp"],
                       direction="backward", allow_exact_matches=False)
f["age_comp_h"] = (f["datetime"] - comp_m["m_time"]).dt.total_seconds().to_numpy() / 3600

q = [0.05, 0.25, 0.5, 0.75, 0.95]
print("\nHours since the machine's last maintenance record, at the moment of each failure:")
print(f["age_any_h"].quantile(q).round(1).to_string())
print("\nHours since the SAME component's last replacement, at failure (per component):")
print(f.groupby("comp")["age_comp_h"].describe(percentiles=[0.05, 0.5, 0.95]).round(1).to_string())
cv = (f.groupby("comp")["age_comp_h"].std() / f.groupby("comp")["age_comp_h"].mean()).round(2)
print("\nCoefficient of variation of component age at failure (small = failures at a nearly fixed age):", cv.to_dict())
print("Most common component ages at failure (in whole days):")
print((f["age_comp_h"].dropna() // 24).astype(int).value_counts().head(8).to_string())

# the same recency measured at ordinary rows (all training rows) for comparison
rec_train = np.expm1(RAW_TRAIN[:, col["hours_since_last_maintenance_log1p"]])
print("\nFor comparison, hours since last maintenance over ALL training rows:")
print(pd.Series(rec_train).quantile(q).round(1).to_string())

In [ ]:
# D4. Positive rate by decile of time-since-last-maintenance (training rows)
rec = pd.Series(np.expm1(RAW_TRAIN[:, col["hours_since_last_maintenance_log1p"]]), name="hours_since_maintenance")
bins = pd.qcut(rec, 10, duplicates="drop")
tab = pd.DataFrame({"bin": bins, "y": ytr_g}).groupby("bin", observed=True)["y"].agg(rows="size", positive_rate_pct=lambda s: 100 * s.mean())
tab["lift_vs_overall"] = (tab["positive_rate_pct"] / (100 * ytr_g.mean())).round(2)
tab["positive_rate_pct"] = tab["positive_rate_pct"].round(2)
print(tab.to_string())
print("\nA risk that rises steadily with age = ordinary wear-out; a risk that spikes inside one narrow band = a deterministic clock.")

## Done — what comes next

Phase 3 (federated learning) must reuse exactly this protocol: the same `fedkg_eval.SetEval` toolkit, the same evaluation sets, thresholds chosen on `validation` only, machine-level cluster bootstrap, and the centralized references above as the upper bound. Every federated experiment should be run on the partitions of the earlier partition notebook (`prep_v2/partitions/`) and for both feature sets and both normalization policies — with the `local` policy evaluated on `test_site_post` / `test_time_late` / `validation`, never on `test_site_full`.